# SIPTA: Diccionario Metodológico y Catálogo Técnico de Indicadores Territoriales

**Fase PDCO**: DEVELOPMENT -> CONTROL | **Sprint**: 2  
**Estándares**: DAMA-BOK (Data Governance & Metadata Management), IEEE 830 / ISO 29148, ISO/IEC 25010  
**Sistema**: Sistema de Indicadores y Priorización Territorial y Alertas Tempranas (SIPTA)

---

## 🎯 Propósito y Alcance del Cuaderno
Este cuaderno constituye el **Diccionario Técnico y Metodológico Oficial de Indicadores de SIPTA**. Su objetivo es:
1. **Explicación Conceptual y Formal**: Documentar de manera rigurosa qué significa cada indicador territorial, su propósito de negocio y la pregunta de política pública que responde.
2. **Formulación Matemática en $\LaTeX$**: Presentar la ecuación matemática formal, variables de numerador y denominador, factores de escala y polaridad analítica (**Directa** vs **Inversa**).
3. **Fichas Técnicas de los 12 Dominios Sectoriales**: Catalogar exhaustivamente los indicadores calculados en Demografía, Salud, Educación, Movilidad, Ambiente, Infraestructura, Finanzas, Vulnerabilidad Social, Seguridad, Servicios Públicos, Empleo y Participación Ciudadana.
4. **Articulación con los 5 Escenarios del IPT**: Explicar qué indicadores alimentan el IPT Base y las variantes de sensibilidad.
5. **Guía de Interpretación para Alertas Tempranas**: Proveer matrices de umbrales críticos (*🔴 Rojo, 🟠 Naranja, 🟡 Amarillo, 🟢 Verde*) para la toma de decisiones presupuestales y focalización de inversión social.
6. **Exportación de Metadatos DAMA-BOK**: Actualizar y persistir el catálogo en `reports/inventory/diccionario_indicadores_sipta.csv`.


## 1. Configuración del Entorno y Carga de Módulos


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().resolve()
if ROOT.name in ["04_modeling", "notebooks"]:
    ROOT = ROOT.parents[1] if ROOT.name == "04_modeling" else ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8

print(f"[OK] Entorno de Diccionario de Indicadores listo. Directorio raíz: {ROOT}")


## 2. Catálogo Maestro de Indicadores Calculados por Dominio

A continuación se estructura el inventario completo de los indicadores calculados en el pipeline territorial de SIPTA:


In [ ]:
catalogo_indicadores = [
    # 1. Demografía
    {
        "Codigo": "DEM-001",
        "Nombre": "Densidad Poblacional",
        "Dominio": "Demografía",
        "Entidad_Fuente": "SDP / DANE",
        "Unidad": "hab/km²",
        "Formula_LaTeX": r"\text{Densidad} = \frac{\text{Población}}{\text{Área km}^2}",
        "Polaridad_IPT": "Informativo / Contexto",
        "Poblacion_Objetivo": "Población total proyectada",
        "Descripcion": "Concentración de habitantes por unidad de superficie territorial.",
    },
    {
        "Codigo": "DEM-002",
        "Nombre": "Población Total Estimada",
        "Dominio": "Demografía",
        "Entidad_Fuente": "DANE Proyecciones",
        "Unidad": "Habitantes",
        "Formula_LaTeX": r"P_i = \sum \text{Habitantes Censados}",
        "Polaridad_IPT": "Denominador Per Cápita",
        "Poblacion_Objetivo": "Población general",
        "Descripcion": "Volumen poblacional base para la normalización de indicadores por habitante.",
    },
    # 2. Salud
    {
        "Codigo": "SAL-001",
        "Nombre": "Sedes IPS por 10.000 Habitantes",
        "Dominio": "Salud",
        "Entidad_Fuente": "SDS / REPS",
        "Unidad": "sedes/10k hab",
        "Formula_LaTeX": r"t_{\text{salud}} = \frac{\text{Sedes IPS Registradas}}{\text{Población}} \times 10\,000",
        "Polaridad_IPT": "Inversa (Carencia = 1 - Norm)",
        "Poblacion_Objetivo": "Población total",
        "Descripcion": "Disponibilidad relativa de establecimientos prestadores de salud (Dimensión 2 IPT).",
    },
    {
        "Codigo": "SAL-002",
        "Nombre": "Camas Hospitalarias por 10.000 Habitantes",
        "Dominio": "Salud",
        "Entidad_Fuente": "SDS / SaluData",
        "Unidad": "camas/10k hab",
        "Formula_LaTeX": r"t_{\text{camas}} = \frac{\text{Camas Totales}}{\text{Población}} \times 10\,000",
        "Polaridad_IPT": "Inversa (Carencia = 1 - Norm)",
        "Poblacion_Objetivo": "Población total",
        "Descripcion": "Capacidad asistencial instalada de camas hospitalarias.",
    },
    # 3. Educación
    {
        "Codigo": "EDU-001",
        "Nombre": "Oferta de Cupos Escolares por 1.000 hab (5-17 años)",
        "Dominio": "Educación",
        "Entidad_Fuente": "SED / SIMAT",
        "Unidad": "cupos/1k hab escolar",
        "Formula_LaTeX": r"t_{\text{edu}} = \frac{\text{Oferta Regular Cupos}}{\text{Población 5 a 17 años}} \times 1\,000",
        "Polaridad_IPT": "Inversa (Carencia = 1 - Norm)",
        "Poblacion_Objetivo": "Niños y jóvenes en edad escolar (5-17 años)",
        "Descripcion": "Capacidad del sistema educativo público frente a la demanda potencial (Dimensión 1 IPT).",
    },
    {
        "Codigo": "EDU-002",
        "Nombre": "Puntaje Promedio Saber 11",
        "Dominio": "Educación",
        "Entidad_Fuente": "ICFES / SED",
        "Unidad": "Puntos (0-500)",
        "Formula_LaTeX": r"\overline{P}_{\text{Saber11}} = \frac{1}{N} \sum_{i=1}^N P_i",
        "Polaridad_IPT": "Inversa (Carencia = 1 - Norm)",
        "Poblacion_Objetivo": "Estudiantes de grado 11",
        "Descripcion": "Nivel de logro académico y calidad educativa media.",
    },
    {
        "Codigo": "EDU-003",
        "Nombre": "Tasa de Deserción Escolar Anual",
        "Dominio": "Educación",
        "Entidad_Fuente": "SED",
        "Unidad": "%",
        "Formula_LaTeX": r"\%_{\text{desercion}} = \frac{\text{Estudiantes Retirados}}{\text{Matrícula Inicial}} \times 100",
        "Polaridad_IPT": "Directa (Alerta Temprana)",
        "Poblacion_Objetivo": "Estudiantes matriculados",
        "Descripcion": "Porcentaje de estudiantes que abandonan el sistema escolar antes de culminar el año.",
    },
    # 4. Movilidad
    {
        "Codigo": "MOV-001",
        "Nombre": "Densidad de Estaciones Troncales TransMilenio",
        "Dominio": "Movilidad",
        "Entidad_Fuente": "TransMilenio S.A.",
        "Unidad": "estaciones/km²",
        "Formula_LaTeX": r"d_{\text{est}} = \frac{\text{Estaciones Troncales}}{\text{Área km}^2}",
        "Polaridad_IPT": "Inversa (Carencia = 1 - Norm)",
        "Poblacion_Objetivo": "Usuarios de transporte masivo",
        "Descripcion": "Acceso espacial al sistema troncal de transporte público (Dimensión 3 IPT).",
    },
    {
        "Codigo": "MOV-002",
        "Nombre": "Densidad de Paraderos Zonales SITP",
        "Dominio": "Movilidad",
        "Entidad_Fuente": "TransMilenio S.A.",
        "Unidad": "paraderos/km²",
        "Formula_LaTeX": r"d_{\text{par}} = \frac{\text{Paraderos SITP}}{\text{Área km}^2}",
        "Polaridad_IPT": "Inversa (Carencia = 1 - Norm)",
        "Poblacion_Objetivo": "Usuarios de transporte zonal",
        "Descripcion": "Cobertura espacial de paraderos del sistema zonal SITP en el territorio.",
    },
    {
        "Codigo": "MOV-003",
        "Nombre": "Tiempo Promedio de Viaje Laboral",
        "Dominio": "Movilidad",
        "Entidad_Fuente": "SDM / Encuesta Movilidad",
        "Unidad": "Minutos",
        "Formula_LaTeX": r"\overline{T}_{\text{viaje}} = \frac{1}{N} \sum T_i",
        "Polaridad_IPT": "Directa (Pérdida de Bienestar)",
        "Poblacion_Objetivo": "Trabajadores ocupados",
        "Descripcion": "Duración media del desplazamiento habitual hacia el lugar de trabajo.",
    },
    # 5. Ambiente
    {
        "Codigo": "AMB-001",
        "Nombre": "Densidad de Conflictos Ambientales SAC",
        "Dominio": "Ambiente",
        "Entidad_Fuente": "SDA / SAC",
        "Unidad": "conflictos/km²",
        "Formula_LaTeX": r"d_{\text{conf}} = \frac{\text{Conflictos SAC Registrados}}{\text{Área km}^2}",
        "Polaridad_IPT": "Directa (Vulnerabilidad = Norm)",
        "Poblacion_Objetivo": "Territorio y ecosistemas",
        "Descripcion": "Afectaciones socio-ambientales y pasivos ecológicos registrados (Dimensión 4 IPT).",
    },
    # 6. Infraestructura
    {
        "Codigo": "INF-001",
        "Nombre": "Parques IDRD por 10.000 Habitantes",
        "Dominio": "Infraestructura",
        "Entidad_Fuente": "IDRD / DADEP",
        "Unidad": "parques/10k hab",
        "Formula_LaTeX": r"t_{\text{parques}} = \frac{\text{Parques IDRD}}{\text{Población}} \times 10\,000",
        "Polaridad_IPT": "Inversa (Carencia = 1 - Norm)",
        "Poblacion_Objetivo": "Población general",
        "Descripcion": "Disponibilidad de parques y espacios recreativos públicos (Dimensión 5 IPT).",
    },
    # 7. Vulnerabilidad
    {
        "Codigo": "VUL-001",
        "Nombre": "Tasa de Vendedores Informales RIVI por 10.000 Hab.",
        "Dominio": "Vulnerabilidad",
        "Entidad_Fuente": "IPES / RIVI",
        "Unidad": "vendedores/10k hab",
        "Formula_LaTeX": r"t_{\text{rivi}} = \frac{\text{Vendedores Informales Promedio}}{\text{Población}} \times 10\,000",
        "Polaridad_IPT": "Directa (Vulnerabilidad = Norm)",
        "Poblacion_Objetivo": "Población en informalidad",
        "Descripcion": "Incidencia de empleo informal en espacio público (Dimensión 6 IPT).",
    },
    # 8. Seguridad
    {
        "Codigo": "SEG-001",
        "Nombre": "Cuadrantes Policiales por 10.000 Habitantes",
        "Dominio": "Seguridad",
        "Entidad_Fuente": "MEBOG / SCJ",
        "Unidad": "cuadrantes/10k hab",
        "Formula_LaTeX": r"t_{\text{cuad}} = \frac{\text{Cuadrantes Policiales}}{\text{Población}} \times 10\,000",
        "Polaridad_IPT": "Inversa (Carencia = 1 - Norm)",
        "Poblacion_Objetivo": "Población general",
        "Descripcion": "Cobertura de patrullaje policial preventivo (Dimensión 7 IPT).",
    },
    {
        "Codigo": "SEG-002",
        "Nombre": "Tasa de Homicidios por 100.000 Habitantes",
        "Dominio": "Seguridad",
        "Entidad_Fuente": "MEBOG / SCJ",
        "Unidad": "homicidios/100k hab",
        "Formula_LaTeX": r"t_{\text{hom}} = \frac{\text{Homicidios Anuales}}{\text{Población}} \times 100\,000",
        "Polaridad_IPT": "Directa (Alerta Violencia Letal)",
        "Poblacion_Objetivo": "Población general",
        "Descripcion": "Incidencia de violencia letal extrema en el territorio.",
    },
    # 9. Finanzas
    {
        "Codigo": "FIN-001",
        "Nombre": "Inversión FDL Per Cápita",
        "Dominio": "Finanzas",
        "Entidad_Fuente": "SDP / Confis / FDL",
        "Unidad": "Millones COP / hab",
        "Formula_LaTeX": r"t_{\text{fdl}} = \frac{\text{Presupuesto Ejecutado FDL}}{\text{Población}}",
        "Polaridad_IPT": "Informativo / Contraste de Inversión",
        "Poblacion_Objetivo": "Población local",
        "Descripcion": "Inversión pública local ejecutada por habitante.",
    },
    {
        "Codigo": "FIN-002",
        "Nombre": "Porcentaje de Ejecución Presupuestal FDL",
        "Dominio": "Finanzas",
        "Entidad_Fuente": "Sec. Gobierno",
        "Unidad": "%",
        "Formula_LaTeX": r"\%_{\text{ejec}} = \frac{\text{Presupuesto Ejecutado}}{\text{Presupuesto Aprobado}} \times 100",
        "Polaridad_IPT": "Directa (Eficiencia Administrativa)",
        "Poblacion_Objetivo": "Presupuesto distrital",
        "Descripcion": "Capacidad de ejecución del gasto de inversión por alcaldía local.",
    },
    # 10. Servicios Públicos
    {
        "Codigo": "PUB-001",
        "Nombre": "Índice de Riesgo de la Calidad del Agua (IRCA)",
        "Dominio": "Servicios Públicos",
        "Entidad_Fuente": "EAAB / Superservicios",
        "Unidad": "Puntos (0-100)",
        "Formula_LaTeX": r"\text{IRCA} = \sum \text{Ensayos No Conformidad}",
        "Polaridad_IPT": "Directa (Riesgo Sanitario)",
        "Poblacion_Objetivo": "Suscriptores residenciales",
        "Descripcion": "Evaluación del riesgo fisicoquímico y microbiológico del agua para consumo.",
    },
    {
        "Codigo": "PUB-002",
        "Nombre": "Cobertura de Acueducto EAAB",
        "Dominio": "Servicios Públicos",
        "Entidad_Fuente": "EAAB / SSPD",
        "Unidad": "%",
        "Formula_LaTeX": r"\%_{\text{acueducto}} = \frac{\text{Suscriptores Conectados}}{\text{Total Viviendas}} \times 100",
        "Polaridad_IPT": "Inversa (Carencia = 1 - Norm)",
        "Poblacion_Objetivo": "Hogares de la localidad",
        "Descripcion": "Porcentaje de predios formalmente conectados a la red matriz de acueducto.",
    },
    # 11. Empleo
    {
        "Codigo": "EMP-001",
        "Nombre": "Conmutación Laboral Externa hacia el Centro Ampliado (%)",
        "Dominio": "Empleo y Economía",
        "Entidad_Fuente": "DANE / SDM",
        "Unidad": "% de ocupados",
        "Formula_LaTeX": r"\%_{\text{conmut}} = \frac{\text{Ocupados que trabajan fuera}}{\text{Total Ocupados}} \times 100",
        "Polaridad_IPT": "Informativo / Demanda Movilidad",
        "Poblacion_Objetivo": "Población ocupada",
        "Descripcion": "Grado de dependencia laboral externa respecto al centro urbano consolidado.",
    },
    # 12. Participación
    {
        "Codigo": "PAR-001",
        "Nombre": "Peticiones, Quejas y Reclamos (PQR) por 10.000 Hab.",
        "Dominio": "Participación Ciudadana",
        "Entidad_Fuente": "Sec. General / SDQS",
        "Unidad": "PQR / 10k hab",
        "Formula_LaTeX": r"t_{\text{pqr}} = \frac{\text{Total PQR Recibidas}}{\text{Población}} \times 10\,000",
        "Polaridad_IPT": "Directa (Demanda/Inconformidad)",
        "Poblacion_Objetivo": "Ciudadanía activa",
        "Descripcion": "Volumen de demandas, quejas de malla vial y solicitudes ciudadanas registradas.",
    },
]

df_catalogo = pd.DataFrame(catalogo_indicadores)
display(df_catalogo[["Codigo", "Nombre", "Dominio", "Entidad_Fuente", "Unidad", "Polaridad_IPT"]])


## 3. Articulación de los Indicadores con los 5 Escenarios de IPT

El sistema vincula cada indicador del diccionario a su función dentro de las fórmulas del IPT:

```
┌────────────────────────────────────────────────────────────────────────┐
│                    ESTRUCTURA METODOLÓGICA DEL IPT                     │
├──────────────────────────┬──────────────────────┬──────────────────────┤
│ Indicador Fuente         │ Dimensión IPT        │ Polaridad Aplicada   │
├──────────────────────────┼──────────────────────┼──────────────────────┤
│ EDU-001 (Cupos 5-17)     │ dim_educacion        │ Inversa (1 - Norm)   │
│ SAL-001 (Sedes IPS)      │ dim_salud            │ Inversa (1 - Norm)   │
│ MOV-001 + MOV-002        │ dim_movilidad        │ Inversa (1 - Norm)   │
│ AMB-001 (Conflictos SAC) │ dim_ambiente         │ Directa (Norm)       │
│ INF-001 (Parques IDRD)   │ dim_infraestructura  │ Inversa (1 - Norm)   │
│ VUL-001 (Vendedores RIVI)│ dim_vulnerabilidad   │ Directa (Norm)       │
│ SEG-001 (Cuadrantes)     │ dim_seguridad        │ Inversa (1 - Norm)   │
└──────────────────────────┴──────────────────────┴──────────────────────┘
```

### Fórmulas de los 5 Escenarios:
1. **IPT Base (7D)**: $\text{IPT}_{\text{Base}} = \left( \frac{1}{7} \sum_{d=1}^7 s_d \right) \times 100$
2. **IPT Rangos (7D)**: $\text{IPT}_{\text{Rangos}} = \left( \frac{1}{7} \sum_{d=1}^7 \text{PercentileRank}(x_d) \right) \times 100$
3. **IPT Sin Parques (6D)**: $\text{IPT}_{\text{SinParques}} = \left( \frac{1}{6} \sum_{d \neq \text{Inf}} s_d \right) \times 100$
4. **IPT Sin RIVI (6D)**: $\text{IPT}_{\text{SinRIVI}} = \left( \frac{1}{6} \sum_{d \neq \text{Vul}} s_d \right) \times 100$
5. **IPT Sin Proxies (5D)**: $\text{IPT}_{\text{SinProxies}} = \left( \frac{1}{5} \sum_{d \in \{ \text{Edu, Sal, Mov, Amb, Seg} \}} s_d \right) \times 100$


## 4. Distribución Empírica de los Indicadores en las 20 Localidades

Se visualiza el comportamiento empírico de las variables en Bogotá D.C.:


In [ ]:
curated_dir = ROOT / "data" / "curated"
master_ind_path = curated_dir / "master_indicadores_territoriales.csv"

if master_ind_path.exists():
    df_ind = pd.read_csv(master_ind_path)
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 1. Densidad Poblacional
    sns.histplot(df_ind['densidad_poblacional'], kde=True, ax=axes[0, 0], color='#0275d8')
    axes[0, 0].set_title("Distribución: Densidad Poblacional (hab/km²)", fontweight='bold')
    
    # 2. Saber 11
    if 'puntaje_promedio_saber_11' in df_ind.columns:
        sns.histplot(df_ind['puntaje_promedio_saber_11'], kde=True, ax=axes[0, 1], color='#5cb85c')
        axes[0, 1].set_title("Distribución: Puntaje Promedio Saber 11", fontweight='bold')
        
    # 3. Tasa Homicidios
    if 'tasa_homicidios_por_100k_hab_calc' in df_ind.columns:
        sns.histplot(df_ind['tasa_homicidios_por_100k_hab_calc'], kde=True, ax=axes[1, 0], color='#d9534f')
        axes[1, 0].set_title("Distribución: Homicidios por 100k Hab.", fontweight='bold')
        
    # 4. Parques por 10k Hab
    if 'parques_por_10k_hab' in df_ind.columns:
        sns.histplot(df_ind['parques_por_10k_hab'], kde=True, ax=axes[1, 1], color='#f0ad4e')
        axes[1, 1].set_title("Distribución: Parques por 10k Hab.", fontweight='bold')
        
    plt.tight_layout()
    plt.show()


## 5. Matriz de Umbrales Críticos para Alertas Tempranas y Política Pública

Para apoyar la toma de decisiones presupuestales de los Fondos de Desarrollo Local (FDL) y la Secretaría Distrital de Integración Social (SDIS), se define una matriz de semaforización:

| Nivel de Alerta | Rango IPT / Score | Significado Territorial | Acción Recomendada |
|---|---|---|---|
| **🔴 Rojo (Crítica)** | $IPT \ge 60$ o Score Carencia $\ge 0.75$ | Déficit agudo y vulnerabilidad multidimensional severa | Intervención prioritaria inmediata de recursos FDL y programas SDIS |
| **🟠 Naranja (Alta)** | $45 \le IPT < 60$ o Score $0.50 \le s < 0.75$ | Carencia sectorial significativa en dimensiones clave | Focalización presupuestal y monitoreo trimestral |
| **🟡 Amarillo (Media)** | $30 \le IPT < 45$ o Score $0.25 \le s < 0.50$ | Cobertura media con oportunidades de optimización | Mantenimiento preventivo de infraestructura y servicios |
| **🟢 Verde (Baja)** | $IPT < 30$ o Score $s < 0.25$ | Alta disponibilidad de oferta y baja vulnerabilidad relativa | Sostenibilidad operativa y transferencia de buenas prácticas |


In [ ]:
# Carga de la priorización final para diagnóstico de alertas
path_priorizacion = curated_dir / "ipt_priorizacion_localidades.csv"

if path_priorizacion.exists():
    df_prio = pd.read_csv(path_priorizacion)
    
    def clasificar_semaforo(ipt_val):
        if ipt_val >= 60:
            return "🔴 Crítica (Roja)"
        elif ipt_val >= 45:
            return "🟠 Alta (Naranja)"
        elif ipt_val >= 30:
            return "🟡 Media (Amarilla)"
        else:
            return "🟢 Baja (Verde)"
            
    df_prio["Semaforo_Alerta"] = df_prio["IPT_MULTIDIMENSIONAL"].apply(clasificar_semaforo)
    
    print("============================================================")
    print("ESTADO DE ALERTAS TEMPRANAS POR LOCALIDAD (SIPTA)")
    print("============================================================")
    display(df_prio[["codigo_localidad", "localidad", "IPT_MULTIDIMENSIONAL", "ranking_consenso", "nivel_prioridad_consenso", "Semaforo_Alerta"]])


## 6. Exportación del Diccionario de Indicadores (DAMA-BOK)

Se exporta el catálogo formal de indicadores a `reports/inventory/diccionario_indicadores_sipta.csv` para gobernanza y consulta técnica.


In [ ]:
inv_dir = ROOT / "reports" / "inventory"
inv_dir.mkdir(parents=True, exist_ok=True)

out_dict = inv_dir / "diccionario_indicadores_sipta.csv"
df_catalogo.to_csv(out_dict, index=False, encoding="utf-8-sig")

print("============================================================")
print("DICCIONARIO DE INDICADORES EXPORTADO EXITOSAMENTE")
print(f"  * Ruta: {out_dict}")
print(f"  * Total de Indicadores Documentados: {len(df_catalogo)}")
print("============================================================")
